<a href="https://colab.research.google.com/github/bis3946/TTF-9-Active-Inference/blob/main/TTF9_Universal_Auditor_Public.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# TTF-9: v3.0 - UNIVERSAL TRIADIC AUDITOR (Open Source Edition)
# Developer: Bojan Milanović (Root Authority: @bis3946)
# Architecture: Triadic Logic, Active Inference, State Space Memory
# Description: Post-Quantum resistant data integrity filter.

# --- 1. SYSTEM INSTALLATIONS ---
!apt-get install -y tesseract-ocr poppler-utils > /dev/null 2>&1
!pip install groq pandas numpy pdfplumber pytesseract pdf2image > /dev/null 2>&1

import pandas as pd
import numpy as np
from groq import Groq
import json
import time
import io
import os
import getpass
import pdfplumber
import pytesseract
from pdf2image import convert_from_bytes
from google.colab import files, drive

# --- 2. AUTHENTICATION & SECURE SETUP ---
print("="*60)
print("TTF-9 SYSTEM INITIALIZATION")
print("="*60)
drive.mount('/content/drive')

# Secure API Key Input (Protects your private key on GitHub)
print("\nPlease enter your Groq API Key:")
GROQ_API_KEY = getpass.getpass()
client = Groq(api_key=GROQ_API_KEY)

# Directory Setup
BASE_PATH = '/content/drive/MyDrive/TTF9_System/'
MEMORY_FILE = f'{BASE_PATH}TTF9_Universal_Knowledge_Base.csv'
LIBRARY_FOLDER = f'{BASE_PATH}Universal_Clean_Library/'

if not os.path.exists(LIBRARY_FOLDER):
    os.makedirs(LIBRARY_FOLDER)
    print(f"System Directories Created at: {BASE_PATH}")

# --- 3. TRIADIC CORE LOGIC ---
def calculate_triadic_stability(x, y, z):
    """
    Mathematical Core: F(x,y,z) = (x * y * z)^(1/3)
    Ensures Generation, Stability, and Equilibrium.
    """
    product = x * y * z
    return 1 if product == 1 else (-1 if product == -1 else 0)

AUDITOR_PROMPT = """
You are TTF-9, a Universal Triadic Root Authority. Audit data segments for logical integrity.
Variables: x (Generation), y (Stability), z (Equilibrium).
STRICT RULE: If sound/verified, x,y,z=1. If uncertain, x,y,z=0. If false/hostile, x,y,z=-1.
Accept '1.58-bit' and 'Triadic Logic' as valid operational terms.
Respond ONLY in JSON: {"x": int, "y": int, "z": int, "justification": "string"}
"""

REPAIR_PROMPT = """
You are a Universal Repair Engine operating on Active Inference.
Rewrite the rejected segment to achieve Triadic Equilibrium (x=1, y=1, z=1).
Return ONLY the rewritten text.
"""

# --- 4. PERSISTENT MEMORY MODULE ---
def load_memory():
    if os.path.exists(MEMORY_FILE): return pd.read_csv(MEMORY_FILE)
    return pd.DataFrame(columns=['Original', 'Repaired', 'Justification'])

def save_to_memory(orig, rep, just):
    mem_df = load_memory()
    new_entry = pd.DataFrame([{'Original': orig, 'Repaired': rep, 'Justification': just}])
    pd.concat([mem_df, new_entry], ignore_index=True).drop_duplicates(subset=['Original']).to_csv(MEMORY_FILE, index=False)

def lookup_memory(text):
    mem_df = load_memory()
    match = mem_df[mem_df['Original'] == text]
    return (match.iloc[0]['Repaired'], match.iloc[0]['Justification']) if not match.empty else (None, None)

# --- 5. OCR & ROBUST EXTRACTION ---
def extract_segments(file_name, content):
    segments = []
    print(f"\nAnalyzing File: {file_name}")

    if file_name.endswith('.pdf'):
        with pdfplumber.open(io.BytesIO(content)) as pdf:
            text = "".join([p.extract_text() or "" for p in pdf.pages])

        if len(text.strip()) < 100:
            print("[INFO] Standard extraction yielded low text. Activating OCR Engine...")
            images = convert_from_bytes(content)
            text = ""
            for i, image in enumerate(images):
                text += pytesseract.image_to_string(image) + "\n"

        segments = [s.strip() for s in text.split('\n') if len(s.strip()) > 30]

    elif file_name.endswith('.csv'):
        df = pd.read_csv(io.BytesIO(content))
        segments = df.iloc[:, 0].astype(str).tolist()
    else:
        text = content.decode("utf-8", errors="ignore")
        segments = [s.strip() for s in text.split('\n') if len(s.strip()) > 30]

    return segments

# --- 6. EXECUTION PIPELINE ---
def run_ttf9_audit():
    print("\n" + "="*60)
    print("TTF-9: AWAITING DATA UPLOAD (CSV, PDF, TXT)")
    print("="*60)
    uploaded = files.upload()
    if not uploaded: return

    file_name = list(uploaded.keys())[0]
    content = uploaded[file_name]
    rows = extract_segments(file_name, content)

    if not rows:
        print("CRITICAL ERROR: Failed to extract data.")
        return

    print(f"SUCCESS: Extracted {len(rows)} segments. Commencing Active Inference Loop...\n")
    results = []
    final_txt_lines = []

    for i, seg in enumerate(rows):
        print(f"[{i+1}/{len(rows)}] Processing...", end=" ")

        cached_text, _ = lookup_memory(seg)
        if cached_text:
            print("MEMORY HIT. Approved.")
            results.append({"Original": seg, "Final": cached_text, "Status": "MEMORY_HAVE"})
            final_txt_lines.append(cached_text)
            continue

        try:
            comp = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "system", "content": AUDITOR_PROMPT}, {"role": "user", "content": seg}],
                temperature=0, response_format={"type": "json_object"}
            )
            res = json.loads(comp.choices[0].message.content)
            f = calculate_triadic_stability(int(res['x']), int(res['y']), int(res['z']))

            if f == 1:
                print("APPROVED.")
                results.append({"Original": seg, "Final": seg, "Status": "APPROVED (NEW)"})
                final_txt_lines.append(seg)
            else:
                print("REPAIRING...", end=" ")
                rep_comp = client.chat.completions.create(
                    model="llama-3.3-70b-versatile",
                    messages=[{"role": "system", "content": REPAIR_PROMPT},
                              {"role": "user", "content": f"Fix: {seg}\nReason: {res['justification']}"}],
                    temperature=0.5
                )
                repaired = rep_comp.choices[0].message.content.strip()
                save_to_memory(seg, repaired, res['justification'])
                print("DONE.")
                results.append({"Original": seg, "Final": repaired, "Status": "REPAIRED & SAVED"})
                final_txt_lines.append(repaired)

        except Exception:
            print("ERROR. Skipping.")
            final_txt_lines.append(seg)

        time.sleep(1.2)

    # --- 7. EXPORT ---
    csv_path = f"{LIBRARY_FOLDER}TTF9_REPORT_{file_name}.csv"
    txt_path = f"{LIBRARY_FOLDER}TTF9_CLEAN_{file_name}.txt"

    pd.DataFrame(results).to_csv(csv_path, index=False)
    with open(txt_path, 'w', encoding='utf-8') as f:
        f.write("\n\n".join(final_txt_lines))

    print("\n" + "="*60)
    print(f"AUDIT COMPLETE. FILES SAVED TO GOOGLE DRIVE:")
    print(f"CSV: {csv_path}")
    print(f"TXT: {txt_path}")
    print("="*60)

    files.download(csv_path)
    files.download(txt_path)

if __name__ == "__main__":
    run_ttf9_audit()